# M2 · BiLSTM + FastText

Second model of the hierarchy of increasing multilingual capacity used in the paper.

| Model | Representation | Notebook |
|---|---|---|
| M1 | Bag-of-words + feed-forward network | `M1_bow.ipynb` |
| M2 | BiLSTM + frozen FastText vectors | `M2_bilstm_fasttext.ipynb` |
| M3 | mBERT (`bert-base-multilingual-cased`), full fine-tuning | `M3_mbert.ipynb` |
| M4 | XLM-RoBERTa (`xlm-roberta-base`), full fine-tuning | `M4_xlmr.ipynb` |

Architecture adapted from Isnain, Sihabuddin and Suyanto (2020): one BiLSTM layer with 200 units per
direction over frozen `cc.{pt,en,id,kk}.300` FastText vectors (Grave et al., 2018). The vectors are
monolingual and not aligned across languages.

| Code | Corpus | Language | Platform | n |
|---|---|---|---|---|
| PT1 | HateBR (Vargas et al., 2022) | Portuguese | Instagram, political accounts | 7,000 |
| PT2 | OffComBR-3 (de Pelle & Moreira, 2017) | Portuguese | news portal comments | 1,033 |
| PT3 | ToLD-Br (Leite et al., 2020), stratified subsample | Portuguese | Twitter, general | 7,000 |
| EN | Davidson et al. (2017) | English | Twitter, general | 24,783 |
| ID | Alfina et al. (2017) | Indonesian | Twitter, political | 713 |
| KZ | Bolatbek et al. (2025) | Kazakh | YouTube, Telegram, VK | 10,190 |

**Inputs.** `protocolo_pibic.py` and the `splits/` folder in `BASE`, and the six corpus files in `BASE/dados/`
(see `data/corpora/README.md`). On Colab, `BASE` is `MyDrive/pibic`; elsewhere it is `./pibic`. M2 also needs the translation cache written by M1, so run M1 first.
The FastText language code for Kazakh is `kk`, not `kz`.

## 1. Imports and configuration

In [ ]:
import os, re, string, json, gzip, urllib.request, datetime, random
import pandas as pd
import numpy as np
import warnings; warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

STOPWORDS = {
    'pt': set(stopwords.words('portuguese')),
    'en': set(stopwords.words('english')),
    'id': set(stopwords.words('indonesian')),
    'kz': set(),
}

# ── M2 hyperparameters (fixed; never tuned against the test set) ────────────
EMBED_DIM     = 300     # cc.XX.300
HIDDEN_DIM    = 200     # Isnain et al. (2020)
DROPOUT       = 0.3
LR            = 0.001
BATCH_SIZE    = 32
MAX_EPOCHS    = 20
PATIENCE      = 3
POS_W_MAX     = 10.0
MIN_COUNT     = 1       # with pre-trained embeddings, rare words still have a vector
MIN_VAL_CALIB = 100     # training-time threshold policy
TH_CROSS      = 0.5
EMB_TRAINABLE = False   # frozen embeddings
PAD_IDX, UNK_IDX = 0, 1
print("Config M2 OK")

In [ ]:
import time
from contextlib import contextmanager

TEMPOS = {}
_T0_NOTEBOOK = time.time()

def _fmt(s):
    m, s = divmod(s, 60); h, m = divmod(m, 60)
    if h: return f"{int(h)}h{int(m):02d}m{s:04.1f}s"
    if m: return f"{int(m)}m{s:04.1f}s"
    return f"{s:.1f}s"

@contextmanager
def cronometro(nome):
    '''Usage: with cronometro("section name"): ...block...'''
    t0 = time.time()
    print(f"⏱  [{nome}] starting...")
    yield
    dt = time.time() - t0
    TEMPOS[nome] = dt
    total = time.time() - _T0_NOTEBOOK
    print(f"⏱  [{nome}] done in {_fmt(dt)}  (notebook total: {_fmt(total)})")

print("Timer ready — model: M2 BiLSTM+FastText")


## 2. NLLB cache (shared with M1)

Point `CACHE_DIR` to the folder used by M1. If every direction shows `[ok]`, this notebook never calls NLLB.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = '/content/drive/MyDrive/pibic/traducoes'
except ImportError:
    CACHE_DIR = os.path.abspath('./pibic/traducoes')
os.makedirs(CACHE_DIR, exist_ok=True)

MT_CHECKPOINT = 'facebook/nllb-200-distilled-600M'
REGISTRO = {
    'modelo': 'M2 BiLSTM + FastText cc.300',
    'mt_checkpoint': MT_CHECKPOINT,
    'embeddings': 'cc.{pt,en,id,kk}.300 congelados',
    'data': str(datetime.date.today()),
}
print(REGISTRO)

## 3. Corpora, binarisation and trivial floor (same as M1)

In [ ]:
# ═══════════════ PROTOCOL SHARED BY THE FOUR MODELS ═══════════════
# Loads protocolo_pibic.py, which holds the ToLD-Br (PT3) loader, the shared
# partitions, prediction persistence and metrics. Place the file in BASE
# before running.
import os, sys, json
from datetime import datetime

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE = '/content/drive/MyDrive/pibic'
except Exception:
    BASE = os.path.abspath('./pibic')

DATA_DIR  = os.path.join(BASE, 'dados')
OUT_DIR   = os.path.join(BASE, 'resultados')
SPLIT_DIR = os.path.join(BASE, 'splits')
for d in (DATA_DIR, OUT_DIR, SPLIT_DIR):
    os.makedirs(d, exist_ok=True)

sys.path.insert(0, BASE)
import protocolo_pibic as PP

def caminho_dado(nome):
    # Accepts the file in the data directory or in the current directory.
    for c in (os.path.join(DATA_DIR, nome), nome):
        if os.path.exists(c):
            return c
    raise FileNotFoundError(
        f'{nome} not found in {DATA_DIR} or in the current directory.')

TAG = datetime.now().strftime('%Y%m%d_%H%M')
print(f'BASE = {BASE}\nTAG  = {TAG}')

def parse_offcombr3(path):
    texts, labels = [], []
    with open(path, encoding='utf-8', errors='ignore') as f:
        in_data = False
        for line in f:
            line = line.strip()
            if not in_data:
                in_data = line.lower().startswith('@data'); continue
            if not line: continue
            lab, _, resto = line.partition(',')
            texto = resto.strip().strip("'")
            if lab in ('yes', 'no') and texto:
                texts.append(texto); labels.append(1 if lab == 'yes' else 0)
    return texts, labels


# ═══════════════ SIX CORPORA ═══════════════
#
# Three Portuguese corpora give six ordered Portuguese→Portuguese cells, so the
# domain cost does not rest on a single pair of corpora.
#
# Leite, Silva, Bontcheva & Scarton (2020), AACL-IJCNLP. License: CC BY-SA 4.0.
# Download ToLD-BR.csv from https://github.com/joaoaleite/ToLD-Br into DATA_DIR.

LIMIAR_ANOTADORES_PT3 = 2      # simple majority; see protocolo_pibic.carregar_toldbr
N_MAX_PT3 = 7000               # matches the size of HateBR; None keeps all 21,000

DATASETS = {}

df = pd.read_csv(caminho_dado('HateBR.csv'))
DATASETS['PT1'] = {'texts': [PP.sanitizar(t) for t in df['comentario'].astype(str)],
                   'labels': df['label_final'].astype(int).tolist(),
                   'lang': 'pt', 'nome': 'HateBR (PT1)',
                   'dominio': 'político / Instagram'}

t, l = parse_offcombr3(caminho_dado('OffComBR3.arff'))
DATASETS['PT2'] = {'texts': [PP.sanitizar(x) for x in t], 'labels': l,
                   'lang': 'pt', 'nome': 'OffComBR-3 (PT2)',
                   'dominio': 'comentários de portal de notícias'}

t, l, META_PT3 = PP.carregar_toldbr(caminho_dado('ToLD-BR.csv'),
                                    limiar_anotadores=LIMIAR_ANOTADORES_PT3,
                                    n_max=N_MAX_PT3)
DATASETS['PT3'] = {'texts': t, 'labels': l,
                   'lang': 'pt', 'nome': 'ToLD-Br (PT3)',
                   'dominio': 'Twitter, geral', 'meta': META_PT3}

df = pd.read_csv(caminho_dado('DataIngles.csv'))
DATASETS['EN'] = {'texts': [PP.sanitizar(x) for x in df['tweet'].astype(str)],
                  'labels': (df['class'] != 2).astype(int).tolist(),
                  'lang': 'en', 'nome': 'Davidson (EN)', 'dominio': 'Twitter, geral'}

bruto = open(caminho_dado('DataIndonesio.txt'), encoding='utf-8',
             errors='ignore').read().replace('\r\n', '\n').replace('\r', '\n')
linhas = [x for x in bruto.split('\n') if '\t' in x][1:]
DATASETS['ID'] = {'texts': [PP.sanitizar(x.split('\t', 1)[1]) for x in linhas],
                  'labels': [1 if x.split('\t', 1)[0].strip() == 'HS' else 0
                             for x in linhas],
                  'lang': 'id', 'nome': 'Alfina (ID)', 'dominio': 'Twitter, político'}

df = pd.read_csv(caminho_dado('DataKazake.csv'))
DATASETS['KZ'] = {'texts': [PP.sanitizar(x) for x in df['message'].astype(str)],
                  'labels': (df['label'] != 0).astype(int).tolist(),
                  'lang': 'kz', 'nome': 'Bolatbek (KZ)',
                  'dominio': 'YouTube / Telegram / VK'}

SIGLAS = [s for s in PP.SIGLAS if s in DATASETS]
for k in SIGLAS:
    ds = DATASETS[k]
    ds['sigla'] = k
    ds['prev_pos'] = float(np.mean(ds['labels']))
    ds['baseline'] = PP.baseline_majoritaria(ds['labels'])

# Trivial floor per corpus (macro-F1 of the majority-class classifier).
BASELINE_TRIVIAL = {k: DATASETS[k]['baseline'] for k in SIGLAS}

print(f"\n{'Code':<6} {'Corpus':<20} {'Lang':<5} {'N':>7} {'%pos':>7} {'trivial':>9}")
print('-' * 60)
for k in SIGLAS:
    ds = DATASETS[k]
    print(f"{k:<6} {ds['nome']:<20} {ds['lang']:<5} {len(ds['labels']):>7,} "
          f"{ds['prev_pos']*100:>6.1f}% {ds['baseline']:>9.3f}")

# PT2 (19.6%) and PT3 (19.3%) have almost the same prevalence, so the PT2 x PT3
# pair is a domain contrast without a base-rate confounder.


## 4. Shared partitions

In [ ]:

# ═══════════════ PARTITIONS SHARED BY M1–M4 ═══════════════
# Loaded from SPLIT_DIR. If each notebook drew its own split, differences between
# models would mix model effects with partition effects.
CHAVES = {k: [PP.chave_norm(t) for t in DATASETS[k]['texts']] for k in SIGLAS}

for k in SIGLAS:
    ds = DATASETS[k]
    outros = {o: CHAVES[o] for o in SIGLAS if o != k}
    ds['idx_tr'], ds['idx_val'], ds['idx_te'] = PP.obter_particoes(
        k, ds['texts'], ds['labels'], SPLIT_DIR, chaves_outros_corpora=outros)

PP.conferir_particoes(SIGLAS, SPLIT_DIR)

print(f"\n{'Corpus':<6} {'train':>8} {'val':>7} {'test':>7} {'%pos test':>12}")
print('-' * 44)
for k in SIGLAS:
    ds = DATASETS[k]
    y = np.array(ds['labels'])
    print(f"{k:<6} {len(ds['idx_tr']):>8,} {len(ds['idx_val']):>7,} "
          f"{len(ds['idx_te']):>7,} {y[ds['idx_te']].mean()*100:>11.1f}%")


In [ ]:

# ═══════════════ RESULTS AND PREDICTIONS LOG ═══════════════
# Every cell of the matrix saves the predicted PROBABILITIES to .npz, so AUC and
# the three threshold regimes (R1 fixed 0.5, R2 prevalence-matched, R3 oracle)
# can be recomputed without retraining.
MODELO_ID = 'M2'
PRED_DIR = PP.dir_predicoes(OUT_DIR, MODELO_ID, TAG)
RESULTADOS = []

def registrar_celula(k_tr, k_te, cenario, semente, p, y, limiar_politica=0.5,
                     verboso=True):
    PP.salvar_predicoes(PRED_DIR, k_tr, k_te, semente, p, y, cenario=cenario)
    m   = PP.metricas(y, p, limiar_politica)
    m05 = PP.metricas(y, p, 0.5)
    reg = PP.tres_regimes(y, p, DATASETS[k_te]['prev_pos'])
    linha = dict(modelo=MODELO_ID, treino=k_tr, teste=k_te, cenario=cenario,
                 semente=semente, grupo=PP.grupo_cenario(k_tr, k_te),
                 mesma_lingua=(DATASETS[k_tr]['lang'] == DATASETS[k_te]['lang']),
                 n_teste=int(len(y)), piso=DATASETS[k_te]['baseline'],
                 macro_f1=m['macro_f1'], limiar=m['limiar'],
                 macro_f1_05=m05['macro_f1'], auc_roc=m05['auc_roc'],
                 auc_pr=m05['auc_pr'], f1_pos_05=m05['f1_pos'],
                 f1_neg_05=m05['f1_neg'], rec_pos_05=m05['rec_pos'],
                 taxa_predicao_positiva_05=m05['taxa_predicao_positiva'])
    linha.update(reg)
    linha['R2_sobre_piso'] = linha['R2_prevalencia'] - linha['piso']
    RESULTADOS.append(linha)
    if verboso:
        marca = 'ok' if linha['R2_sobre_piso'] > 0 else 'BELOW FLOOR'
        print(f"    {k_tr:>4} -> {k_te:<4} [{cenario:<12}] s{semente}  "
              f"R1={linha['macro_f1_05']:.3f}  R2={linha['R2_prevalencia']:.3f}  "
              f"AUC={linha['auc_roc']:.3f}  {marca}")
    return linha


## 5. Preprocessing and reading the NLLB cache

In [ ]:
def preprocess(text, lang='pt'):
    stop = STOPWORDS.get(lang, set())
    tok_lang = {'pt':'portuguese','en':'english','id':'indonesian'}.get(lang,'english')
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|@\w+|#\w+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    try:
        tokens = word_tokenize(text, language=tok_lang)
    except Exception:
        tokens = text.split()
    return [t for t in tokens if t not in stop and len(t) >= 2
            and sum(c.isalpha() for c in t)/len(t) >= 0.7]

for key, ds in DATASETS.items():
    ds['tokenized'] = [preprocess(t, ds['lang']) for t in ds['texts']]
print("Preprocessing of the original texts done.")

def obter_traducao(key_dataset, tgt_lang):
    '''Read a translation from the NLLB cache written by M1. Does NOT translate; warns if missing.'''
    ds = DATASETS[key_dataset]
    path = os.path.join(CACHE_DIR, f"{key_dataset}__para_{tgt_lang}.csv")
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing cache: {path}\n"
            f"Create it with the M1 notebook (or copy it from Drive) before running this scenario.")
    df = pd.read_csv(path)
    assert len(df) == len(ds['texts']), f"cache inconsistente: {path}"
    return df['traduzido'].astype(str).tolist()

LANGS = sorted({ds['lang'] for ds in DATASETS.values()})
print("\nNLLB cache status:")
for k in DATASETS:
    for lg in LANGS:
        if DATASETS[k]['lang'] == lg: continue
        ok = os.path.exists(os.path.join(CACHE_DIR, f"{k}__para_{lg}.csv"))
        print(f"  {'[ok]' if ok else '[MISSING]'} {k:<8} → {lg}")

## 6. Pre-trained FastText vectors (cc.XX.300), filtered loading

The `.vec.gz` files have about 1.2 GB per language. They are streamed and only the words that occur in
the corpora (originals and translations) are kept, in a filtered `.npz` per language.

In [ ]:
FT_CODES = {'pt': 'pt', 'en': 'en', 'id': 'id', 'kz': 'kk'}
FT_URL   = 'https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.{code}.300.vec.gz'
FT_DIR   = os.path.join(BASE, 'fasttext')
os.makedirs(FT_DIR, exist_ok=True)

def palavras_necessarias_por_lingua():
    '''Collect the words that can appear in each language:
       original texts in that language + translations INTO that language.'''
    need = {lg: set() for lg in LANGS}
    for k, ds in DATASETS.items():
        for doc in ds['tokenized']:
            need[ds['lang']].update(doc)
    for k in DATASETS:
        for lg in LANGS:
            if DATASETS[k]['lang'] == lg: continue
            path = os.path.join(CACHE_DIR, f"{k}__para_{lg}.csv")
            if os.path.exists(path):
                trad = pd.read_csv(path)['traduzido'].astype(str).tolist()
                for t in trad:
                    need[lg].update(preprocess(t, lg))
    return need

def carregar_fasttext(lang, need_words):
    '''Return a word -> vector dict for need_words only. Uses an .npz cache.'''
    code_ = FT_CODES[lang]
    npz_path = os.path.join(FT_DIR, f'cc.{code_}.300.filtrado.npz')
    if os.path.exists(npz_path):
        z = np.load(npz_path, allow_pickle=True)
        vecs = dict(zip(z['words'].tolist(), z['matrix']))
        faltam = need_words - set(vecs)
        if not faltam or len(faltam)/max(len(need_words),1) < 0.5:
            print(f"  [{lang}] filtered cache: {len(vecs):,} words")
            return vecs
        print(f"  [{lang}] filtered cache out of date — reloading from .vec.gz")
    gz_path = os.path.join(FT_DIR, f'cc.{code_}.300.vec.gz')
    if not os.path.exists(gz_path):
        url = FT_URL.format(code=code_)
        print(f"  [{lang}] downloading {url} (~1.2 GB)...")
        urllib.request.urlretrieve(url, gz_path)
    print(f"  [{lang}] filtering {len(need_words):,} words from .vec.gz...")
    vecs = {}
    with gzip.open(gz_path, 'rt', encoding='utf-8', errors='ignore') as f:
        next(f)  # header
        for line in f:
            sp = line.rstrip().split(' ')
            if sp[0] in need_words:
                vecs[sp[0]] = np.asarray(sp[1:], dtype=np.float32)
                if len(vecs) == len(need_words): break
    np.savez_compressed(npz_path,
                        words=np.array(list(vecs.keys()), dtype=object),
                        matrix=np.stack(list(vecs.values())) if vecs else np.zeros((0,300)))
    cob = len(vecs)/max(len(need_words),1)
    print(f"  [{lang}] FastText coverage: {cob:.1%} ({len(vecs):,}/{len(need_words):,})")
    return vecs

print("Collecting the vocabulary needed for each language...")
with cronometro("carregamento_fasttext"):
    NEED = palavras_necessarias_por_lingua()
    FT = {lg: carregar_fasttext(lg, NEED[lg]) for lg in LANGS}

## 7. Vocabulary, sequences, model and training

In [ ]:
from collections import Counter

def build_vocab(tokenized_train, min_count=MIN_COUNT):
    counts = Counter(t for doc in tokenized_train for t in doc)
    w2i = {'<PAD>': PAD_IDX, '<UNK>': UNK_IDX}
    for w, c in sorted(counts.items()):
        if c >= min_count:
            w2i[w] = len(w2i)
    return w2i

def build_emb_matrix(w2i, lang):
    vecs = FT[lang]
    M = np.zeros((len(w2i), EMBED_DIM), dtype=np.float32)
    achadas = []
    for w, i in w2i.items():
        if w in vecs:
            M[i] = vecs[w]; achadas.append(i)
    if achadas:
        M[UNK_IDX] = M[achadas].mean(axis=0)   # UNK = mean of the known vectors
    cob = len(achadas)/max(len(w2i)-2,1)
    print(f"    embedding coverage ({lang}): {cob:.1%}")
    return M

def encode_pad(tokens, w2i, max_len):
    ids = [w2i.get(t, UNK_IDX) for t in tokens]
    return (ids[:max_len] if len(ids) >= max_len
            else ids + [PAD_IDX]*(max_len-len(ids)))

def make_seqs(tokenized, w2i, max_len):
    return np.array([encode_pad(t, w2i, max_len) for t in tokenized], dtype=np.int64)

class BiLSTM(nn.Module):
    def __init__(self, vocab_size, emb_matrix):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=PAD_IDX)
        self.embedding.weight = nn.Parameter(torch.tensor(emb_matrix))
        self.embedding.weight.requires_grad = EMB_TRAINABLE
        self.bilstm = nn.LSTM(EMBED_DIM, HIDDEN_DIM, num_layers=1,
                              batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(HIDDEN_DIM*2, 1)
    def forward(self, x):                       # returns LOGITS
        emb = self.dropout(self.embedding(x))
        _, (h, _) = self.bilstm(emb)
        out = torch.cat((h[-2], h[-1]), dim=1)
        return self.fc(self.dropout(out)).squeeze(1)

class SeqDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

def treinar_bilstm(X_tr, y_tr, X_val, y_val, emb_matrix, seed=SEED, verbose=True):
    torch.manual_seed(seed); np.random.seed(seed)
    n_pos = max(int(y_tr.sum()), 1); n_neg = max(len(y_tr)-int(y_tr.sum()), 1)
    pw = torch.tensor(min(n_neg/n_pos, POS_W_MAX), dtype=torch.float32, device=DEVICE)
    if verbose: print(f"    pos_weight={pw.item():.2f}")
    model = BiLSTM(emb_matrix.shape[0], emb_matrix).to(DEVICE)
    crit  = nn.BCEWithLogitsLoss(pos_weight=pw)
    opt   = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
    tr_loader = DataLoader(SeqDS(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
    Xv = torch.tensor(X_val, dtype=torch.long).to(DEVICE)
    yv = torch.tensor(y_val, dtype=torch.float32).to(DEVICE)
    best_loss, best_state, pat = np.inf, None, 0
    for ep in range(1, MAX_EPOCHS+1):
        model.train()
        for Xb, yb in tr_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(Xb), yb)
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vl = crit(model(Xv), yv).item()
        if vl < best_loss - 1e-4:
            best_loss, pat = vl, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            pat += 1
            if pat >= PATIENCE:
                if verbose: print(f"    Early stopping at epoch {ep}")
                break
    model.load_state_dict(best_state)
    return model

def probs(model, X, bs=256):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            xb = torch.tensor(X[i:i+bs], dtype=torch.long).to(DEVICE)
            out.append(torch.sigmoid(model(xb)).cpu().numpy())
    return np.concatenate(out)

def melhor_threshold(model, X_val, y_val, min_n=MIN_VAL_CALIB):
    # QUANTILE grid instead of a linear grid: with scores concentrated near 0 and 1,
    # a linear grid can miss the useful threshold (same procedure as M3/M4). This only
    # affects the `macro_f1` column; R1/R2/R3 come from the shared protocol.
    if len(y_val) < min_n:
        return 0.5
    p = probs(model, X_val)
    return PP.escolher_limiar(p, np.asarray(y_val).astype(int))

def avaliar(y_true, y_pred, titulo):
    mf1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    m = {'acc': accuracy_score(y_true, y_pred),
         'prec': precision_score(y_true, y_pred, zero_division=0),
         'rec': recall_score(y_true, y_pred, zero_division=0),
         'f1': f1_score(y_true, y_pred, zero_division=0),
         'macro_f1': mf1, 'cm': confusion_matrix(y_true, y_pred).tolist()}
    print(f"  {titulo:<48} Macro-F1: {mf1:.3f}")
    return m

print("Model and training functions defined.")

## 8. Base training: one model per corpus and seed

Maximum length = 95th percentile of the token lengths of the training partition. Out-of-vocabulary
words receive the mean of the known vectors.

In [ ]:
SEMENTES = [42, 43, 44]

def fixar_semente(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

MODELOS = {}      # (corpus, seed) -> dict

with cronometro("treino_modelos_base"):
    for semente in SEMENTES:
        for k in SIGLAS:
            ds = DATASETS[k]
            print(f"\n=== M2 base: {ds['nome']} · seed {semente} ===")
            fixar_semente(semente)
            tok = ds['tokenized']
            y = np.array(ds['labels'], dtype=np.float32)
            tok_tr = [tok[i] for i in ds['idx_tr']]
            w2i = build_vocab(tok_tr)
            max_len = max(10, int(np.percentile([len(t) for t in tok_tr], 95)))
            emb = build_emb_matrix(w2i, ds['lang'])
            X_tr = make_seqs(tok_tr, w2i, max_len)
            X_val = make_seqs([tok[i] for i in ds['idx_val']], w2i, max_len)
            y_val = y[ds['idx_val']]
            print(f"  train={len(X_tr):,} | val={len(X_val):,} | vocab={len(w2i):,} | MAX_LEN={max_len}")
            # Pass seed=semente explicitly. With the default (seed=SEED), treinar_bilstm
            # would call torch.manual_seed(42) internally and override fixar_semente()
            # above, making the three seeds bit-for-bit identical.
            model = treinar_bilstm(X_tr, y[ds['idx_tr']], X_val, y_val, emb,
                                   seed=semente)
            th = melhor_threshold(model, X_val, y_val.astype(int))
            MODELOS[(k, semente)] = {'model': model, 'w2i': w2i,
                                     'max_len': max_len, 'th_intra': th}
            print(f"  in-corpus threshold (val, n={len(y_val)}): {th:.2f}")
print("\nM2 base models trained.")

## 9. In-corpus, cross-domain and cross-lingual cells (threshold 0.5 on transfer cells)

In [ ]:
def probs_m2(model_dict, tokenized):
    X = make_seqs(tokenized, model_dict['w2i'], model_dict['max_len'])
    return np.asarray(probs(model_dict['model'], X), dtype=np.float64).flatten()

with cronometro("intra_e_cross"):
    for semente in SEMENTES:
        print(f"\n─── seed {semente} ───")
        for k_tr in SIGLAS:
            m = MODELOS[(k_tr, semente)]
            for k_te in SIGLAS:
                ds_te = DATASETS[k_te]
                # ALWAYS the test partition (see M1).
                tok_te = [ds_te['tokenized'][i] for i in ds_te['idx_te']]
                y_te = np.array(ds_te['labels'])[ds_te['idx_te']]
                p = probs_m2(m, tok_te)
                lim = m['th_intra'] if k_tr == k_te else 0.5
                registrar_celula(k_tr, k_te, 'base', semente, p, y_te, lim)

# ── Check that the seeds actually produced different runs ───────────────────
# Guards against a silent failure: three identical seeds go unnoticed in any
# table of means but cancel the noise estimate.
_chk = (pd.DataFrame(RESULTADOS)
        .query("cenario == 'base'")
        .groupby(['treino', 'teste'])['R2_prevalencia'].nunique())
_iguais = int((_chk == 1).sum())
print(f'\nCells with the same value in all 3 seeds: {_iguais} of {len(_chk)}')
if _iguais == len(_chk):
    raise RuntimeError(
        'The three seeds produced identical results. Check that '
        '`seed=semente` is passed to treinar_bilstm(). Without variation '
        'across seeds there is no noise estimate and the reported '
        'uncertainty margin would be wrong.')
print('✓ The seeds varied — the noise estimate is valid.')

## 10. Translate-test (reuses the base models; reads translations from the M1 cache)

In [ ]:
def executar_mt(k_tr, k_te, semente):
    ds_tr, ds_te = DATASETS[k_tr], DATASETS[k_te]
    m = MODELOS[(k_tr, semente)]
    trad = obter_traducao(k_te, ds_tr['lang'])
    idx = ds_te['idx_te']
    tok_trad = [preprocess(trad[i], ds_tr['lang']) for i in idx]
    y_te = np.array(ds_te['labels'])[idx]
    registrar_celula(k_tr, k_te, 'mt', semente, probs_m2(m, tok_trad), y_te, 0.5)

with cronometro("translate_test_mt"):
    for semente in SEMENTES:
        print(f"\n─── MT · seed {semente} ───")
        for k_tr in SIGLAS:
            for k_te in SIGLAS:
                if k_tr != k_te and DATASETS[k_tr]['lang'] != DATASETS[k_te]['lang']:
                    executar_mt(k_tr, k_te, semente)

## 11. Translate-train

One model per cell, trained on the translated source with FastText vectors of the target language,
evaluated on the original target test set.

In [ ]:
def executar_tt(k_tr, k_te, semente):
    ds_tr, ds_te = DATASETS[k_tr], DATASETS[k_te]
    lang_alvo = ds_te['lang']
    fixar_semente(semente)
    trad = obter_traducao(k_tr, lang_alvo)
    tok_trad = [preprocess(t, lang_alvo) for t in trad]
    y = np.array(ds_tr['labels'], dtype=np.float32)

    tok_tr = [tok_trad[i] for i in ds_tr['idx_tr']]
    w2i = build_vocab(tok_tr)
    max_len = max(10, int(np.percentile([len(t) for t in tok_tr], 95)))
    emb = build_emb_matrix(w2i, lang_alvo)
    X_tr = make_seqs(tok_tr, w2i, max_len)
    X_val = make_seqs([tok_trad[i] for i in ds_tr['idx_val']], w2i, max_len)
    y_val = y[ds_tr['idx_val']]
    model = treinar_bilstm(X_tr, y[ds_tr['idx_tr']], X_val, y_val, emb,
                           seed=semente)

    md = {'model': model, 'w2i': w2i, 'max_len': max_len}
    idx = ds_te['idx_te']
    tok_te = [ds_te['tokenized'][i] for i in idx]
    y_te = np.array(ds_te['labels'])[idx]
    registrar_celula(k_tr, k_te, 'tt', semente, probs_m2(md, tok_te), y_te, 0.5)

EXECUTAR_TT = True
with cronometro("translate_train_tt"):
    if EXECUTAR_TT:
        for semente in SEMENTES:
            print(f"\n─── TT · seed {semente} ───")
            for k_tr in SIGLAS:
                for k_te in SIGLAS:
                    if k_tr != k_te and DATASETS[k_tr]['lang'] != DATASETS[k_te]['lang']:
                        executar_tt(k_tr, k_te, semente)

## 12. Summary tables

In [ ]:
print("\n" + "="*50)
print("RUN TIME SUMMARY")
print("="*50)
for nome, dt in TEMPOS.items():
    print(f"  {nome:<28} {_fmt(dt)}")
print("-"*50)
print(f"  {'TOTAL (notebook)':<28} {_fmt(time.time()-_T0_NOTEBOOK)}")


In [ ]:

# ═══════════════ TRANSLATE-TEST x TRANSLATE-TRAIN ═══════════════
# Scenario names: 'base' (zero-shot), 'mt' (translate-test), 'tt' (translate-train).
# Each value is the MEAN over the three seeds. R2 (prevalence-matched threshold)
# is the main column; R1 (threshold 0.5) is shown next to it.

def pegar(k_tr, k_te, cen, col='R2_prevalencia'):
    r = [x for x in RESULTADOS
         if x['treino'] == k_tr and x['teste'] == k_te and x['cenario'] == cen]
    return float(np.mean([x[col] for x in r])) if r else np.nan

fmt = lambda v: f'{v:7.3f}' if not np.isnan(v) else '      —'

for col, rot in [('R1_fixo', 'R1 · fixed threshold 0.5'),
                 ('R2_prevalencia', 'R2 · prevalence-matched  ← main')]:
    print(f'\n{"="*72}\n  {rot}\n{"="*72}')
    print(f"{'Train':<7}{'Test':<7}{'Trivial':>8}{'zero-shot':>11}"
          f"{'MT':>8}{'TT':>8}{'Δ MT':>8}{'Δ TT':>8}")
    print('-' * 72)
    for k_tr in SIGLAS:
        for k_te in SIGLAS:
            if k_tr == k_te or DATASETS[k_tr]['lang'] == DATASETS[k_te]['lang']:
                continue
            base = pegar(k_tr, k_te, 'base', col)
            mt   = pegar(k_tr, k_te, 'mt',   col)
            tt   = pegar(k_tr, k_te, 'tt',   col)
            print(f'{k_tr:<7}{k_te:<7}{BASELINE_TRIVIAL[k_te]:>8.3f}'
                  f'{fmt(base)}{fmt(mt)}{fmt(tt)}{fmt(mt-base)}{fmt(tt-base)}')

print('\nPositive Δ = translation helps relative to direct transfer.')
print('Always compare against the trivial floor of the TARGET: a gain over zero-shot')
print('means nothing if the whole cell is below the floor.')


## 13. Results persistence (same format as M3/M4)

In [ ]:

# ═══════════════ PERSISTENCE (same format as M3/M4) ═══════════════
DF = pd.DataFrame(RESULTADOS)

AGG = (DF[DF.cenario == 'base']
       .groupby(['treino', 'teste', 'grupo', 'mesma_lingua', 'piso', 'n_teste'],
                as_index=False)
       .agg(macro_f1_05_media=('macro_f1_05', 'mean'),
            macro_f1_05_dp=('macro_f1_05', 'std'),
            R1_fixo=('R1_fixo', 'mean'),
            R2_prevalencia=('R2_prevalencia', 'mean'),
            R2_dp=('R2_prevalencia', 'std'),
            R3_oraculo=('R3_oraculo', 'mean'),
            custo_calibracao=('custo_calibracao', 'mean'),
            auc_media=('auc_roc', 'mean'), auc_dp=('auc_roc', 'std'),
            n_sementes=('semente', 'nunique')))
AGG['R2_sobre_piso'] = AGG['R2_prevalencia'] - AGG['piso']

# Sanity check: R3 is an upper bound and must dominate R1 and R2 in every cell.
viol = AGG[(AGG.R3_oraculo < AGG.R1_fixo - 1e-6) |
           (AGG.R3_oraculo < AGG.R2_prevalencia - 1e-6)]
print(f'Violations of "R3 is an upper bound": {len(viol)} of {len(AGG)}')

print('\n── Aggregated matrix (regime R2) ──')
print(AGG.sort_values(['treino', 'teste'])[
    ['treino', 'teste', 'grupo', 'piso', 'R1_fixo', 'R2_prevalencia',
     'R3_oraculo', 'R2_sobre_piso', 'auc_media', 'macro_f1_05_dp']
    ].to_string(index=False, float_format=lambda v: f'{v:6.3f}'))

print('\n── Means by group ──')
print(AGG.groupby('grupo')[['R1_fixo', 'R2_prevalencia', 'R3_oraculo',
                            'custo_calibracao', 'auc_media']]
        .agg(['mean', 'count']).round(3).to_string())

SAIDA = dict(
    modelo=MODELO_ID, gerado_em=datetime.now().isoformat(),
    corpora={k: {kk: vv for kk, vv in DATASETS[k].items()
                 if kk not in ('texts', 'labels', 'tokenized',
                               'idx_tr', 'idx_val', 'idx_te')}
             for k in SIGLAS},
    particoes={k: dict(n_treino=len(DATASETS[k]['idx_tr']),
                       n_val=len(DATASETS[k]['idx_val']),
                       n_teste=len(DATASETS[k]['idx_te'])) for k in SIGLAS},
    resultados=RESULTADOS,
    agregado=AGG.to_dict(orient='records'),
    tempos_execucao={k: round(v / 60, 3) for k, v in sorted(TEMPOS.items())}
        if 'TEMPOS' in dir() else {},
)
arq_json = os.path.join(OUT_DIR, f'{MODELO_ID}_resultados_{TAG}.json')
with open(arq_json, 'w', encoding='utf-8') as f:
    json.dump(SAIDA, f, ensure_ascii=False, indent=2, default=float)
AGG.to_csv(os.path.join(OUT_DIR, f'{MODELO_ID}_matriz_{TAG}.csv'), index=False)

print(f'\nSaved:\n  {arq_json}\n  {MODELO_ID}_matriz_{TAG}.csv\n  {PRED_DIR}/')
